# 9.5 Publication Text Analysis - step 5: does research content predict energy use beyond admin data?

In [1]:
# Set up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config
from sklearn.linear_model import LassoCV
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

sys.path.append(str(Path.cwd().parents[0] / "functions"))
from make_ml_comparison_table import make_ml_comparison_table

In [2]:
# Load data

labs = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv",
    keep_default_na=False,
    na_values=[""],
)
publications = pd.read_csv(
    config.PUBLICATON_DATA / 
    "2_Processed" / 
    "publications_matched.csv"
)

lab_topic_shares = pd.read_csv(
    config.PUBLICATON_DATA / 
    "3_Clean" / 
    "lab_topic_shares.csv"
)

topic_names = pd.read_csv(
    config.PUBLICATON_DATA /
    "3_Clean" /
    "topic_top_words.csv"
)

## (1) Rebuild the 9_0 administrative baseline

In [3]:
# Restrict to BL labs with matched publications
bl_data = labs[labs["survey"] == "BL"].copy()
bl_data = bl_data[["labgroupid", "annual_electricity_total", "no_researchers", "faculty", "institute_id"]].copy()

matched_labgroupids = (
    publications["matched_labgroupids"].astype(str).str.split(";").explode().str.strip().unique()
)
bl_data = bl_data[bl_data["labgroupid"].astype(str).isin(matched_labgroupids)].copy()

# Prepare vars for regression
bl_data["log_energy"] = np.log(bl_data["annual_electricity_total"])
bl_data["log_no_researchers"] = np.log(bl_data["no_researchers"])
bl_data["science_faculty"] = np.where(bl_data["faculty"] == "Faculty of Science (MNF)", 1, 0)

# Group small institutes into "Other" categories
institute_counts = bl_data["institute_id"].value_counts()
small_institutes = institute_counts[institute_counts < 3].index
bl_data["institute_grouped"] = np.where(
    (bl_data["institute_id"].isin(small_institutes)) & (bl_data["science_faculty"] == 1),
    "Other_science",
    np.where(
        (bl_data["institute_id"].isin(small_institutes)) & (bl_data["science_faculty"] == 0),
        "Other_nonscience",
        bl_data["institute_id"].astype(str),
    ),
)

# Print summary stats
print(f"Labs: {len(bl_data)}")
print(f"Institute categories: {bl_data['institute_grouped'].nunique()}")

Labs: 95
Institute categories: 12


## (2) Merge topic shares

In [4]:
# Check number of topics
topic_cols = [c for c in lab_topic_shares.columns if c.startswith("topic_")]

# Merge data
bl_data["labgroupid"] = bl_data["labgroupid"].astype(int)
lab_topic_shares["labgroupid"] = lab_topic_shares["labgroupid"].astype(int)
model_data = bl_data.merge(lab_topic_shares, on="labgroupid", how="inner")

# Print summary stats
print(f"Labs with both admin data and topic shares: {len(model_data)}")
print(f"Topic columns: {len(topic_cols)}")

Labs with both admin data and topic shares: 95
Topic columns: 4


## (3) Model A: administrative data only (same as 9_0)

In [5]:
features_admin = pd.get_dummies(
    model_data[["institute_grouped"]], columns=["institute_grouped"], drop_first=True
)
features_admin.insert(0, "log_no_researchers", model_data["log_no_researchers"].values)
X_admin = features_admin.values.astype(float)
y = model_data["log_energy"].values

outer_cv = KFold(n_splits=5, shuffle=True, random_state=0)
reg = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", LassoCV(cv=5, max_iter=5000, random_state=0)),
])

cv_r2_admin = cross_val_score(reg, X_admin, y, cv=outer_cv, scoring="r2")
print(f"Model A (admin only) nested CV R²: {cv_r2_admin.mean():.3f} (+/- {cv_r2_admin.std():.3f})")

Model A (admin only) nested CV R²: 0.485 (+/- 0.043)


## (4) Model B: administrative data + topic shares

In [6]:
features_full = features_admin.copy()
for col in topic_cols:
    features_full[col] = model_data[col].values
X_full = features_full.values.astype(float)

cv_r2_full = cross_val_score(reg, X_full, y, cv=outer_cv, scoring="r2")
print(f"Model B (admin + topics) nested CV R²: {cv_r2_full.mean():.3f} (+/- {cv_r2_full.std():.3f})")
print(f"Model A (admin only) nested CV R²:      {cv_r2_admin.mean():.3f} (+/- {cv_r2_admin.std():.3f})")
print(f"Incremental R² from adding topic shares: {cv_r2_full.mean() - cv_r2_admin.mean():.3f}")

Model B (admin + topics) nested CV R²: 0.531 (+/- 0.049)
Model A (admin only) nested CV R²:      0.485 (+/- 0.043)
Incremental R² from adding topic shares: 0.046


## (5) Which features does model B use?

In [7]:
reg_full_fit = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", LassoCV(cv=5, max_iter=5000, random_state=0)),
])
reg_full_fit.fit(X_full, y)
lasso_full = reg_full_fit.named_steps["lasso"]

coef_df = pd.DataFrame({"feature": features_full.columns, "coefficient": lasso_full.coef_})
coef_df["abs_coef"] = coef_df["coefficient"].abs()
n_nonzero = (coef_df["coefficient"] != 0).sum()
n_topics_nonzero = (coef_df.loc[coef_df["feature"].isin(topic_cols), "coefficient"] != 0).sum()

print(f"Selected alpha: {lasso_full.alpha_:.4f}")
print(f"Total features with non-zero coefficient: {n_nonzero} / {len(features_full.columns)}")
print(f"Of which topic shares: {n_topics_nonzero} / {len(topic_cols)}")
print()
print(coef_df.sort_values("abs_coef", ascending=False).drop(columns="abs_coef").to_string(index=False))

Selected alpha: 0.0233
Total features with non-zero coefficient: 14 / 16
Of which topic shares: 4 / 4

                           feature  coefficient
                log_no_researchers     0.837077
            institute_grouped_1025     0.588512
                           topic_2    -0.475653
                           topic_6     0.268681
                           topic_3    -0.205119
            institute_grouped_1015     0.185887
            institute_grouped_1058    -0.138938
institute_grouped_Other_nonscience    -0.126124
            institute_grouped_1072     0.120832
            institute_grouped_1064     0.116922
            institute_grouped_1074     0.088602
                           topic_0    -0.060144
            institute_grouped_1039     0.021521
            institute_grouped_1037    -0.006357
            institute_grouped_1042    -0.000000
   institute_grouped_Other_science     0.000000


## (6) Summary table for the paper

Standard nested-model comparison format: Model (1) admin-only, Model (2) admin + topics, R² and N, indicator for institute FE. These are LASSO-selected coefficients on standardized features - no significance stars or p-values.

In [8]:
# Fit model A on the full sample too, so both models' coefficients are
# available for the table (model B's full-sample fit was already done above)
reg_admin_fit = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", LassoCV(cv=5, max_iter=5000, random_state=0)),
])
reg_admin_fit.fit(X_admin, y)
admin_coef = dict(zip(features_admin.columns, reg_admin_fit.named_steps["lasso"].coef_))
full_coef = dict(zip(features_full.columns, lasso_full.coef_))

# Topic labels from topic_names
TOPIC_LABELS = dict(zip(topic_names["topic"], topic_names["topic_label"]))
TOPIC_LABELS = {f"topic_{k}": v for k, v in TOPIC_LABELS.items()}

# Selected topics (non-zero coefficients in model B, ordered by coefficient magnitude)
selected_topics = (
    coef_df.loc[coef_df["feature"].isin(topic_cols) & (coef_df["coefficient"] != 0)]
    .sort_values("abs_coef", ascending=False)["feature"]
    .tolist()
)

table = make_ml_comparison_table(
    model_coefs = [admin_coef, full_coef],
    model_names = ["(1) Admin only", "(2) Admin + Topics"],
    n_obs       = [len(model_data), len(model_data)],
    r2          = [cv_r2_admin.mean(), cv_r2_full.mean()],
    keep_vars   = ["log_no_researchers"] + selected_topics,
    var_labels  = {
        "log_no_researchers": "log(no.\\ researchers), std.",
        **{k: v for k, v in TOPIC_LABELS.items()},
    },
    r2_label    = "Cross-validated R$^2$",
    fe_rows     = {"Institute FE": [True, True]},
    decimals    = [2, 2],
    col1_width  = "6cm",
    coln_width  = "3cm",
)

table_path = config.OUTPUT / "11_Publications" / "topic_energy_comparison.tex"
_ = table_path.write_text(table)
print(table)

\begin{tabular}{@{}L{6cm}C{3cm}C{3cm}}
\hline
\addlinespace[0.2cm]
 & (1) Admin only & (2) Admin + Topics \\
\hline
\addlinespace[0.2cm]
log(no.\ researchers), std. & $ 0.92$ & $ 0.84$ \\
\addlinespace[0.1cm]
General/broad & \textemdash & $\llap{-}0.48$ \\
\addlinespace[0.1cm]
Immunology and virology & \textemdash & $ 0.27$ \\
\addlinespace[0.1cm]
Flavour physics & \textemdash & $\llap{-}0.21$ \\
\addlinespace[0.1cm]
Collider physics & \textemdash & $\llap{-}0.06$ \\
\addlinespace[0.1cm]
\addlinespace[0.1cm]
\hline
\addlinespace[0.2cm]
Institute FE & \checkmark & \checkmark \\
Number of observations & $ 95$ & $ 95$ \\
Cross-validated R$^2$ & $ 0.49$ & $ 0.53$ \\
\addlinespace[0.2cm]
\hline
\end{tabular}
